In [ ]:
!pip install ultralytics paho-mqtt opencv-python-headless

In [ ]:
import base64
import json
import cv2
import numpy as np
import paho.mqtt.client as mqtt
from ultralytics import YOLO
import os
import time
from google.colab import files  # Jika dijalankan di Colab
from IPython.display import Image, display  # Untuk menampilkan hasil di Colab
import matplotlib.pyplot as plt

In [ ]:
# ### 1. Muat Model Hasil Training Sendiri
# Model berikut adalah hasil training YOLOv8 pada notebook
# `Roboflow_Model_Yolo_Mask.ipynb` (dataset mask-wearing dari Roboflow).
#
#   "v1" -> yolov8n, imgsz training 640, mAP50=0.849 (validation)
#   "v2" -> yolov8s, imgsz training 960, mAP50=0.828 (validation)
#   "v3" -> yolov8s, imgsz training 960, +763 gambar data tambahan --
#           recall wajah sulit membaik, TAPI bias ke "mask" (salah
#           klasifikasi wajah tanpa masker sebagai pakai masker -- JANGAN
#           dipakai untuk deployment, cuma untuk perbandingan/riwayat)
#   "v4" -> v3 dengan data TAMBAHAN di-rebalance kelasnya -- memperbaiki
#           sebagian besar kasus, tapi bias residual masih muncul di
#           sejumlah foto baru (keseimbangan kelas TOTAL belum dicek)
#   "v5" -> v4 dengan keseimbangan kelas TOTAL (asli+tambahan) dicek dan
#           di-rebalance lebih jauh -- kandidat final berikutnya
MODEL_VERSION = "v5"

MODEL_FILES = {"v1": "best.pt", "v2": "best_v2.pt", "v3": "best_v3.pt", "v4": "best_v4.pt", "v5": "best_v5.pt"}

from google.colab import drive
drive.mount('/content/drive')

model_filename = MODEL_FILES[MODEL_VERSION]
model_path = f'/content/drive/MyDrive/yolo-mask-detection/{model_filename}'
print(f"Memuat model {MODEL_VERSION}: {model_path}")
if MODEL_VERSION == "v3":
    print("PERINGATAN: v3 diketahui bias mengklasifikasi wajah tanpa masker "
          "sebagai 'Pakai Masker'. Gunakan v4/v5 untuk deployment nyata.")

if not os.path.exists(model_path):
    print(f"Model tidak ditemukan di {model_path}.")
    print("Silakan upload file secara manual sebagai alternatif.")
    from google.colab import files as _files
    uploaded = _files.upload()
    if uploaded:
        model_path = list(uploaded.keys())[0]

In [ ]:
# ### 2. Verifikasi Model Tersedia

if not os.path.exists(model_path):
    print(f"Model {model_path} tidak ditemukan. Pastikan file best.pt sudah diunggah.")
else:
    print(f"Model {model_path} ditemukan dan siap digunakan.")

In [ ]:
# ### 3. Konfigurasi MQTT Shiftr.io
MQTT_BROKER = "public.cloud.shiftr.io"
MQTT_PORT = 1883
MQTT_USERNAME = "public"
MQTT_PASSWORD = "public"
INPUT_TOPIC = "stream-ingestion"
OUTPUT_TOPIC = "ai-result"

# ### 4. Inisialisasi Model YOLOv8 (model hasil training sendiri)
model = YOLO(model_path)

In [ ]:
# ### 5. Definisi Fungsi

def decode_base64_image(encoded_data):
    """Decodes a base64 encoded image back to OpenCV format."""
    image_data = base64.b64decode(encoded_data)
    np_arr = np.frombuffer(image_data, np.uint8)
    img = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
    return img

def encode_base64_image(image_path):
    """Encodes an image file to base64 format."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def process_image(image):
    """Runs the YOLOv8 model on the image and returns results."""
    results = model(image)
    detected_objects = []

    for r in results:
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            detected_object = {"xyxy": [x1, y1, x2, y2]}
            detected_objects.append(detected_object)

    return detected_objects

In [ ]:
# ### 6. Unggah Gambar

def upload_image_colab():
    """Upload image file using Google Colab's file upload feature."""
    uploaded = files.upload()  # Mengunggah file menggunakan widget Colab
    if not uploaded:
        print("No file uploaded.")
        return None

    # Ambil file pertama yang diunggah
    for filename in uploaded.keys():
        print(f"Uploaded file: {filename}")
        return filename

In [ ]:
# ### 7. Fungsi Tambahan untuk Pengujian Lokal (versi awal)

def test_local_image(image_path):
    """Test model on a locally uploaded image and publish results."""
    if not os.path.exists(image_path):
        print(f"File {image_path} tidak ditemukan.")
        return

    print(f"Processing image {image_path}...")
    image = cv2.imread(image_path)

    objects = process_image(image)

    print("Detection Results:")
    print(json.dumps({"objects": objects}, indent=2))

    # Simpan hasil deteksi dengan bounding box
    for obj in objects:
        x1, y1, x2, y2 = obj["xyxy"]
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

    output_path = "detection_result.jpg"
    cv2.imwrite(output_path, image)
    print(f"Saved result to {output_path}")

    # Tampilkan gambar hasil deteksi
    display_detected_image(image)

    # Publish the result to MQTT
    publish_detection_result(objects)

def publish_detection_result(objects):
    """Publish object detection results via MQTT."""
    result = json.dumps({"objects": objects})
    client.publish(OUTPUT_TOPIC, result)
    print(f"Published AI result to '{OUTPUT_TOPIC}': {result}")

def display_detected_image(image):
    """Display the detected image with bounding boxes using matplotlib."""
    # Convert BGR image to RGB for displaying in matplotlib
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 8))
    plt.imshow(image_rgb)
    plt.axis('off')  # Hide axes
    plt.show()

In [ ]:
# ### 8. Fungsi Deteksi Masker (versi final, dengan label kategori)
# Konfigurasi final berdasarkan eksperimen di section-section sebelumnya,
# dijalankan di 5 versi model (v1: yolov8n/640; v2: yolov8s/960; v3: +763
# gambar data tambahan; v4: v3 data tambahan di-rebalance; v5: v4 + total
# 105 gambar asli ikut di-rebalance):
# - v5 + imgsz=1280 + conf=0.1 + agnostic_nms=True: satu-satunya versi
#   yang benar mengklasifikasi SEMUA kasus wajah tanpa masker di uji nyata
#   (v3/v4 masih sesekali salah label "Pakai Masker").
# - max_size_ratio=4.0: buang kotak yang luasnya >4x median kotak lain di
#   foto yang sama (kemungkinan besar background/objek, bukan wajah) --
#   dijalankan iteratif, bukan sekali, karena kotak nyasar bisa lebih dari
#   satu dengan ukuran berbeda-beda.
# - dedup_iou=0.3: di conf serendah 0.1, kadang muncul 2 kotak KELAS SAMA
#   yang tumpang tindih pada wajah yang sama (NMS bawaan YOLO pakai
#   threshold default yang longgar, jadi kotak dengan overlap ~30-60%
#   masih lolos sebagai "dua objek berbeda"). Kalau ada 2+ kotak kelas
#   sama yang overlap-nya > dedup_iou, cuma kotak confidence tertinggi
#   yang disimpan.
# Trade-off: imgsz=1280 membuat inferensi ~5-6x lebih lambat dibanding 640
# (diukur: ~120ms vs ~750-2200ms per gambar di Tesla T4).


def _box_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0


def _deduplicate_same_class(detections, iou_threshold=0.3):
    """Kalau 2+ kotak kelas sama tumpang tindih di atas iou_threshold,
    simpan cuma yang confidence-nya paling tinggi."""
    sorted_dets = sorted(detections, key=lambda d: d["confidence"], reverse=True)
    kept = []
    for d in sorted_dets:
        is_duplicate = any(
            d["category"] == k["category"]
            and _box_iou(d["bounding_box"], k["bounding_box"]) > iou_threshold
            for k in kept
        )
        if not is_duplicate:
            kept.append(d)
    return kept


# Fungsi Deteksi Masker
def detect_mask(image, conf=0.1, imgsz=1280, agnostic_nms=True, max_size_ratio=4.0, dedup_iou=0.3):
    """Runs YOLOv8 detection on the input image."""
    results = model(image, conf=conf, imgsz=imgsz, agnostic_nms=agnostic_nms)
    raw_detections = []

    for r in results:
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            class_name = model.names[int(box.cls[0])]
            category = "Pakai Masker" if class_name == "mask" else "Tidak Pakai Masker"
            area = max(1, (x2 - x1) * (y2 - y1))
            raw_detections.append({
                "category": category,
                "bounding_box": [x1, y1, x2, y2],
                "confidence": float(box.conf[0]),
                "_area": area,
            })

    # 1. Buang kotak yang jauh lebih besar dari median (kemungkinan besar
    # bukan wajah). Iteratif -- lihat catatan di atas.
    detections = raw_detections
    while len(detections) >= 3:
        areas = sorted(d["_area"] for d in detections)
        median_area = areas[len(areas) // 2]
        filtered = [d for d in detections if d["_area"] <= median_area * max_size_ratio]
        if len(filtered) == len(detections):
            break
        detections = filtered

    # 2. Buang duplikat kelas-sama yang tumpang tindih
    detections = _deduplicate_same_class(detections, iou_threshold=dedup_iou)

    annotated_image = image.copy()
    for d in detections:
        x1, y1, x2, y2 = d["bounding_box"]
        color = (0, 255, 0) if d["category"] == "Pakai Masker" else (0, 0, 255)
        cv2.rectangle(annotated_image, (x1, y1), (x2, y2), color, 2)

    detections = [
        {"category": d["category"], "bounding_box": d["bounding_box"], "confidence": d["confidence"]}
        for d in detections
    ]

    return detections, annotated_image

# Fungsi Pengujian Gambar Lokal
def test_local_image_new(image_path):
    """Test model on a locally uploaded image and publish results."""
    if not os.path.exists(image_path):
        print(f"File {image_path} tidak ditemukan.")
        return

    print(f"Processing image {image_path}...")
    image = cv2.imread(image_path)

    detections, annotated_image = detect_mask(image)

    print("Detection Results:")
    print(json.dumps({"objects": detections}, indent=2))

    output_path = "detection_result.jpg"
    cv2.imwrite(output_path, annotated_image)
    print(f"Saved result to {output_path}")

    # Tampilkan gambar hasil deteksi
    display_detected_image(annotated_image)

    # Publish the result to MQTT
    publish_detection_result(detections)

In [ ]:
# ### 9. Eksperimen: Bandingkan Confidence Threshold & Image Size
# Beberapa deteksi tidak muncul (terutama wajah kecil/jauh di foto CCTV).
# Coba beberapa kombinasi conf & imgsz untuk melihat pengaruhnya terhadap recall.

def detect_mask_configurable(image, conf=0.25, imgsz=640):
    """Sama seperti detect_mask, tapi conf threshold dan imgsz bisa diatur."""
    results = model(image, conf=conf, imgsz=imgsz)
    detections = []
    annotated_image = image.copy()

    for r in results:
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            class_name = model.names[int(box.cls[0])]
            category = "Pakai Masker" if class_name == "mask" else "Tidak Pakai Masker"
            detections.append({
                "category": category,
                "bounding_box": [x1, y1, x2, y2],
                "confidence": float(box.conf[0])
            })
            color = (0, 255, 0) if category == "Pakai Masker" else (0, 0, 255)
            cv2.rectangle(annotated_image, (x1, y1), (x2, y2), color, 2)

    return detections, annotated_image


def compare_conf_imgsz(image_path, configs=None):
    """
    Bandingkan hasil deteksi dengan beberapa kombinasi conf & imgsz pada gambar yang sama.
    configs: list of dict, misal [{"conf": 0.25, "imgsz": 640}, {"conf": 0.1, "imgsz": 1280}]
    """
    if not os.path.exists(image_path):
        print(f"File {image_path} tidak ditemukan.")
        return

    if configs is None:
        configs = [
            {"conf": 0.25, "imgsz": 640},   # setelan default
            {"conf": 0.10, "imgsz": 640},   # threshold lebih rendah
            {"conf": 0.25, "imgsz": 1280},  # resolusi lebih tinggi
            {"conf": 0.10, "imgsz": 1280},  # kombinasi keduanya
        ]

    image = cv2.imread(image_path)
    results_summary = []

    fig, axes = plt.subplots(1, len(configs), figsize=(6 * len(configs), 6))
    if len(configs) == 1:
        axes = [axes]

    for ax, cfg in zip(axes, configs):
        detections, annotated = detect_mask_configurable(image, conf=cfg["conf"], imgsz=cfg["imgsz"])
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        ax.imshow(annotated_rgb)
        ax.set_title(f"conf={cfg['conf']}, imgsz={cfg['imgsz']}\n{len(detections)} objek terdeteksi")
        ax.axis('off')
        results_summary.append({**cfg, "jumlah_deteksi": len(detections)})

    plt.tight_layout()
    plt.show()

    print("Ringkasan perbandingan:")
    for r in results_summary:
        print(f"  conf={r['conf']}, imgsz={r['imgsz']} -> {r['jumlah_deteksi']} objek terdeteksi")

    return results_summary

In [ ]:
# ### 10. Jalankan Perbandingan
# Upload gambar CCTV yang tadi hasilnya kurang optimal, lalu bandingkan
# beberapa kombinasi conf & imgsz sekaligus.

test_image_name = upload_image_colab()
if test_image_name:
    comparison_results = compare_conf_imgsz(test_image_name)

In [ ]:
# ### 11. Verifikasi Manual: Deteksi Baru di Conf Rendah -- Asli atau Noise?
# Menurunkan conf threshold menambah jumlah deteksi, tapi belum tentu semuanya
# wajah asli (true positive). Cell ini mencari deteksi yang HANYA muncul di
# conf rendah (tidak overlap dengan deteksi di conf tinggi/baseline), lalu
# crop & zoom tiap deteksi baru itu untuk dicek manual satu per satu.

def compute_iou(box1, box2):
    """IoU antara dua bounding box [x1, y1, x2, y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0


def find_new_detections(baseline_dets, comparison_dets, iou_threshold=0.3):
    """Deteksi di comparison_dets yang tidak overlap signifikan dengan baseline_dets."""
    new_dets = []
    for det in comparison_dets:
        box = det["bounding_box"]
        is_new = True
        for base_det in baseline_dets:
            if compute_iou(box, base_det["bounding_box"]) > iou_threshold:
                is_new = False
                break
        if is_new:
            new_dets.append(det)
    return new_dets


def inspect_new_detections(image_path, conf_baseline=0.25, conf_lower=0.1, imgsz=640, iou_threshold=0.3):
    """Bandingkan baseline vs conf rendah, lalu crop & tampilkan khusus deteksi barunya."""
    if not os.path.exists(image_path):
        print(f"File {image_path} tidak ditemukan.")
        return

    image = cv2.imread(image_path)
    baseline_dets, _ = detect_mask_configurable(image, conf=conf_baseline, imgsz=imgsz)
    lower_dets, _ = detect_mask_configurable(image, conf=conf_lower, imgsz=imgsz)
    new_dets = find_new_detections(baseline_dets, lower_dets, iou_threshold=iou_threshold)

    print(f"Baseline (conf={conf_baseline}): {len(baseline_dets)} deteksi")
    print(f"Conf rendah (conf={conf_lower}): {len(lower_dets)} deteksi")
    print(f"Deteksi BARU yang muncul di conf rendah: {len(new_dets)}")

    if not new_dets:
        print("Tidak ada deteksi baru untuk diperiksa.")
        return new_dets

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    n = len(new_dets)
    cols = min(n, 6)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = axes.flatten() if n > 1 else [axes]

    h, w = image_rgb.shape[:2]
    for ax, det in zip(axes, new_dets):
        x1, y1, x2, y2 = det["bounding_box"]
        pad = 15
        crop = image_rgb[max(0, y1 - pad):min(h, y2 + pad), max(0, x1 - pad):min(w, x2 + pad)]
        ax.imshow(crop)
        ax.set_title(f"{det['category']}\nconf={det['confidence']:.2f}", fontsize=9)
        ax.axis('off')

    for ax in axes[len(new_dets):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

    return new_dets

In [ ]:
# ### 12. Jalankan Verifikasi
# Pakai gambar yang sama dengan cell perbandingan sebelumnya.

if test_image_name:
    new_detections = inspect_new_detections(test_image_name, conf_baseline=0.25, conf_lower=0.1, imgsz=640)

In [ ]:
# ### 13. Uji agnostic_nms untuk Deteksi Ganda Antar-Kelas
# Beberapa "deteksi baru" di cell sebelumnya ternyata kotak duplikat pada
# wajah yang SAMA dengan label kelas berbeda (mask vs no-mask). Ini terjadi
# karena NMS default (agnostic_nms=False) cuma menekan tumpang tindih pada
# kelas yang sama. Untuk 2 kelas yang saling eksklusif (satu orang cuma bisa
# salah satu), agnostic_nms=True seharusnya lebih tepat.

def detect_mask_agnostic(image, conf=0.1, imgsz=640, agnostic_nms=False):
    results = model(image, conf=conf, imgsz=imgsz, agnostic_nms=agnostic_nms)
    detections = []
    annotated_image = image.copy()
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            class_name = model.names[int(box.cls[0])]
            category = "Pakai Masker" if class_name == "mask" else "Tidak Pakai Masker"
            detections.append({"category": category, "bounding_box": [x1, y1, x2, y2], "confidence": float(box.conf[0])})
            color = (0, 255, 0) if category == "Pakai Masker" else (0, 0, 255)
            cv2.rectangle(annotated_image, (x1, y1), (x2, y2), color, 2)
    return detections, annotated_image


def compare_agnostic_nms(image_path, conf=0.1, imgsz=640):
    if not os.path.exists(image_path):
        print(f"File {image_path} tidak ditemukan.")
        return
    image = cv2.imread(image_path)

    dets_normal, ann_normal = detect_mask_agnostic(image, conf=conf, imgsz=imgsz, agnostic_nms=False)
    dets_agnostic, ann_agnostic = detect_mask_agnostic(image, conf=conf, imgsz=imgsz, agnostic_nms=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    panels = [
        (f"agnostic_nms=False\n{len(dets_normal)} deteksi", ann_normal),
        (f"agnostic_nms=True\n{len(dets_agnostic)} deteksi", ann_agnostic),
    ]
    for ax, (title, ann) in zip(axes, panels):
        ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    print(f"agnostic_nms=False -> {len(dets_normal)} deteksi")
    print(f"agnostic_nms=True  -> {len(dets_agnostic)} deteksi")
    return dets_normal, dets_agnostic

In [ ]:
# ### 14. Jalankan Uji agnostic_nms
if test_image_name:
    dets_normal, dets_agnostic = compare_agnostic_nms(test_image_name, conf=0.1, imgsz=640)

In [ ]:
# ### 15. Uji Test-Time Augmentation (augment=True)
# Sebagian wajah butuh conf lebih rendah, sebagian lagi butuh imgsz lebih besar
# (detail piksel), dan satu setting tunggal tidak menangkap keduanya sekaligus.
# augment=True menjalankan inferensi multi-scale + flip secara otomatis lalu
# menggabungkan hasilnya -- berpotensi menangkap kedua tipe kasus tanpa harus
# pilih satu kombinasi manual.

def detect_mask_tta(image, conf=0.1, imgsz=640, augment=False, agnostic_nms=True):
    results = model(image, conf=conf, imgsz=imgsz, augment=augment, agnostic_nms=agnostic_nms)
    detections = []
    annotated_image = image.copy()
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            class_name = model.names[int(box.cls[0])]
            category = "Pakai Masker" if class_name == "mask" else "Tidak Pakai Masker"
            detections.append({"category": category, "bounding_box": [x1, y1, x2, y2], "confidence": float(box.conf[0])})
            color = (0, 255, 0) if category == "Pakai Masker" else (0, 0, 255)
            cv2.rectangle(annotated_image, (x1, y1), (x2, y2), color, 2)
    return detections, annotated_image


def compare_tta(image_path, conf=0.1, imgsz=640):
    if not os.path.exists(image_path):
        print(f"File {image_path} tidak ditemukan.")
        return
    image = cv2.imread(image_path)

    dets_normal, ann_normal = detect_mask_tta(image, conf=conf, imgsz=imgsz, augment=False)
    dets_tta, ann_tta = detect_mask_tta(image, conf=conf, imgsz=imgsz, augment=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    panels = [
        (f"augment=False\n{len(dets_normal)} deteksi", ann_normal),
        (f"augment=True (TTA)\n{len(dets_tta)} deteksi", ann_tta),
    ]
    for ax, (title, ann) in zip(axes, panels):
        ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    print(f"augment=False -> {len(dets_normal)} deteksi")
    print(f"augment=True  -> {len(dets_tta)} deteksi")
    return dets_normal, dets_tta

In [ ]:
# ### 16. Jalankan Uji TTA
if test_image_name:
    dets_normal_tta, dets_tta = compare_tta(test_image_name, conf=0.1, imgsz=640)

## Cek Regresi: Foto Close-up Tanpa Masker Salah Diklasifikasi

Ditemukan kasus baru: foto close-up satu orang yang JELAS tidak pakai
masker malah dilabel "Pakai Masker". Ini kebalikan dari masalah
sebelumnya (orang kelewat tidak terdeteksi) -- sekarang orang terdeteksi
tapi salah kelas, dan ini lebih berisiko untuk kasus pemakaian nyata
(sistem bilang aman padahal tidak).

Dugaan: efek samping dataset tambahan v3, yang recall kelas no-mask nya
memang lebih rendah dari v1/v2 (0.65 vs 0.80 di validation). Cell ini
menguji v1/v2/v3 sekaligus di foto yang sama untuk memastikan apakah
ini masalah khusus v3, atau sudah ada sejak awal tapi belum ketahuan
karena belum pernah diuji di foto close-up satu orang.

In [ ]:
# ### 19. Bandingkan v1/v2/v3/v4/v5 pada Foto yang Sama
def load_model_version(version):
    files = {"v1": "best.pt", "v2": "best_v2.pt", "v3": "best_v3.pt", "v4": "best_v4.pt", "v5": "best_v5.pt"}
    path = f'/content/drive/MyDrive/yolo-mask-detection/{files[version]}'
    return YOLO(path)


def compare_model_versions(image_path, conf=0.1, imgsz=1280, versions=("v1", "v2", "v3", "v4", "v5")):
    if not os.path.exists(image_path):
        print(f"File {image_path} tidak ditemukan.")
        return
    image = cv2.imread(image_path)

    fig, axes = plt.subplots(1, len(versions), figsize=(6 * len(versions), 6))
    if len(versions) == 1:
        axes = [axes]

    for ax, version in zip(axes, versions):
        m = load_model_version(version)
        results = m(image, conf=conf, imgsz=imgsz, agnostic_nms=True)
        annotated = image.copy()
        labels = []
        for r in results:
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                class_name = m.names[int(box.cls[0])]
                category = "Pakai Masker" if class_name == "mask" else "Tidak Pakai Masker"
                conf_score = float(box.conf[0])
                labels.append(f"{category} ({conf_score:.2f})")
                color = (0, 255, 0) if category == "Pakai Masker" else (0, 0, 255)
                cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)

        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        ax.imshow(annotated_rgb)
        title = version + ("\n" + "\n".join(labels) if labels else "\n(tidak ada deteksi)")
        ax.set_title(title, fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# ### 20. Jalankan Perbandingan Versi
# Upload foto close-up yang tadi salah diklasifikasi
portrait_image_name = upload_image_colab()
if portrait_image_name:
    compare_model_versions(portrait_image_name, conf=0.1, imgsz=1280)

In [ ]:
# ### 21. Inisialisasi MQTT Client

client = mqtt.Client()
client.username_pw_set(MQTT_USERNAME, MQTT_PASSWORD)

# MQTT callbacks
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("Connected to MQTT Broker!")
        client.subscribe(INPUT_TOPIC)
        print(f"Subscribed to topic '{INPUT_TOPIC}'")
    else:
        print(f"Failed to connect, return code {rc}")

def on_message(client, userdata, message):
    try:
        print(f"Message received on topic: {message.topic}")
        encoded_image = message.payload.decode("utf-8")
        image = decode_base64_image(encoded_image)
        objects = process_image(image)
        result = json.dumps({"objects": objects})
        client.publish(OUTPUT_TOPIC, result)
        print(f"Published AI result to '{OUTPUT_TOPIC}': {result}")
    except Exception as e:
        print(f"Error processing message: {e}")

client.on_connect = on_connect
client.on_message = on_message
client.connect(MQTT_BROKER, MQTT_PORT, keepalive=60)
client.loop_start()

In [ ]:
# ### 22. Mengunggah dan Memproses Gambar Secara Terus-Menerus

def process_until_done():
    while True:
        print("Upload an image for detection (or type 'exit' to stop):")
        uploaded_filename = upload_image_colab()  # Unggah gambar
        if uploaded_filename:
            print(f"Processing {uploaded_filename}...")
            test_local_image_new(uploaded_filename)  # Proses gambar
            print(f"Detection completed for {uploaded_filename}.")
        else:
            print("No image uploaded. Retrying...")

        # Menanyakan apakah pengguna ingin mengunggah gambar lagi atau keluar
        user_input = input("Do you want to upload another image? (y/n): ").lower()
        if user_input != 'y':
            print("Exiting the image upload loop.")
            break
        time.sleep(1)  # Delay kecil agar tidak terus-menerus tanpa jeda

# Panggil fungsi untuk memulai proses
process_until_done()

In [ ]:
# Panggil fungsi untuk memulai proses
process_until_done()

---
*Case study developed as part of the rubythalib.ai AI Engineer Bootcamp (CV Super Class, Object Detection module).*